# AI Programming — Lecture 19
## Further Studies 1: Univariate Patch Transformer for ETTh1

일반 Transformer는 time step 하나를 token 하나로 사용할 수 있습니다.
이번 확장 실습에서는 **여러 time step을 하나의 patch token**으로 묶습니다.

### 설정
```text
Input        : OT only
Lookback     : 96
Prediction   : 24
Patch length : 16
Stride       : 8
Number patches: 11
```

### 핵심 흐름
```text
96 time steps
→ overlapping patches
→ patch projection
→ positional embedding
→ Transformer Encoder
→ multi-horizon residual forecast
```

> Further Studies는 정규 실습을 마친 뒤 선택적으로 진행해도 됩니다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", gpus)


## 1. ETTh1 데이터 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv')

df = pd.read_csv(DATA_PATH)

print("Data path:", DATA_PATH)
print("Shape:", df.shape)
print(df.head())

ot = df[['OT']].values.astype('float32')


## 2. Chronological Split과 Standardization

In [ ]:
LOOKBACK = 96
PRED_LEN = 24

n = len(ot)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_raw = ot[:train_end]
val_raw = ot[train_end:val_end]
test_raw = ot[val_end:]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

print("Train:", train_scaled.shape)
print("Validation:", val_scaled.shape)
print("Test:", test_scaled.shape)


## 3. Forecasting Window 생성

In [ ]:
def create_windows(values, lookback=96, pred_len=24):
    X, Y, Y_res = [], [], []
    total_len = lookback + pred_len

    for i in range(len(values) - total_len + 1):
        window = values[i:i + total_len]
        past = window[:lookback]
        future = window[lookback:]
        baseline = past[-1]

        X.append(past)
        Y.append(future)
        Y_res.append(future - baseline)

    return (
        np.array(X, dtype='float32'),
        np.array(Y, dtype='float32'),
        np.array(Y_res, dtype='float32')
    )

X_train, y_train, y_train_res = create_windows(train_scaled, LOOKBACK, PRED_LEN)
X_val, y_val, y_val_res = create_windows(val_scaled, LOOKBACK, PRED_LEN)
X_test, y_test, y_test_res = create_windows(test_scaled, LOOKBACK, PRED_LEN)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


## 4. Overlapping Patches

`patch_len=16`, `stride=8`로 겹치는 patch를 만듭니다.

각 patch가 하나의 Transformer token 역할을 합니다.

In [ ]:
PATCH_LEN = 16
STRIDE = 8

N_PATCHES = (LOOKBACK - PATCH_LEN) // STRIDE + 1

def create_patches(X, patch_len=16, stride=8):
    patches = []

    for start in range(0, LOOKBACK - patch_len + 1, stride):
        patch = X[:, start:start + patch_len, 0]
        patches.append(patch)

    return np.stack(patches, axis=1).astype('float32')

X_train_patch = create_patches(X_train, PATCH_LEN, STRIDE)
X_val_patch = create_patches(X_val, PATCH_LEN, STRIDE)
X_test_patch = create_patches(X_test, PATCH_LEN, STRIDE)

print("Number of patches:", N_PATCHES)
print("Patched train shape:", X_train_patch.shape)
print("One sample:", X_train_patch[0].shape)


### Patch Layout

96 step을 16-step patch로 나누고 8 step씩 이동하면 11개의 patch가 생성됩니다.

## 5. Learned Positional Embedding

In [ ]:
class LearnedPositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, inputs):
        positions = keras.ops.arange(
            0, keras.ops.shape(inputs)[1], 1
        )
        return inputs + self.position_embedding(positions)


## 6. Transformer Encoder Block

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(ff_dim, activation='relu')
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=None):
        attention_output = self.attention(
            inputs,
            inputs,
            training=training
        )

        x = self.norm1(
            inputs + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.dense2(self.dense1(x))

        return self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )


## 7. Patch Transformer 구성

In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

inputs = keras.Input(shape=(N_PATCHES, PATCH_LEN))

projection_layer = layers.Dense(EMBED_DIM)
x = projection_layer(inputs)

position_layer = LearnedPositionalEmbedding(N_PATCHES, EMBED_DIM)
x = position_layer(x)

encoder1 = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT)
x = encoder1(x)

encoder2 = TransformerEncoder(EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT)
x = encoder2(x)

flatten_layer = layers.Flatten()
x = flatten_layer(x)

output_layer = layers.Dense(PRED_LEN)
outputs = output_layer(x)

model = keras.Model(
    inputs,
    outputs,
    name='univariate_patch_transformer'
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

model.summary()


## 8. Model Training

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_patch,
    y_train_res[..., 0],
    validation_data=(X_val_patch, y_val_res[..., 0]),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Residual MSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()


## 9. Direct Multi-Horizon Evaluation

In [ ]:
pred_res = model.predict(
    X_test_patch,
    batch_size=512,
    verbose=1
)

baseline = X_test[:, -1, 0][:, None]
y_pred = baseline + pred_res

y_test_2d = y_test[..., 0]

norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    y_pred.reshape(-1)
)
norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    y_pred.reshape(-1)
)

y_test_real = scaler.inverse_transform(
    y_test_2d.reshape(-1, 1)
).reshape(y_test_2d.shape)

y_pred_real = scaler.inverse_transform(
    y_pred.reshape(-1, 1)
).reshape(y_pred.shape)

real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)
real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)

print(f'Normalized MSE : {norm_mse:.4f}')
print(f'Normalized MAE : {norm_mae:.4f}')
print(f'MSE (°C²)      : {real_mse:.4f}')
print(f'MAE (°C)       : {real_mae:.4f}')


## 10. Last-Value Baseline

In [ ]:
last_value_pred = np.repeat(
    X_test[:, -1, 0][:, None],
    PRED_LEN,
    axis=1
)

baseline_norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)
baseline_norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)

last_value_real = scaler.inverse_transform(
    last_value_pred.reshape(-1, 1)
).reshape(last_value_pred.shape)

baseline_real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)
baseline_real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)

print('Last-Value Baseline')
print(f'Normalized MSE : {baseline_norm_mse:.4f}')
print(f'Normalized MAE : {baseline_norm_mae:.4f}')
print(f'MSE (°C²)      : {baseline_real_mse:.4f}')
print(f'MAE (°C)       : {baseline_real_mae:.4f}')


## 11. Forecast Example

In [ ]:
sample_idx = 0

past_real = scaler.inverse_transform(
    X_test[sample_idx]
).reshape(-1)

future_real = y_test_real[sample_idx]
pred_real = y_pred_real[sample_idx]

past_x = np.arange(-LOOKBACK + 1, 1)
future_x = np.arange(1, PRED_LEN + 1)

plt.figure(figsize=(10, 4))
plt.plot(past_x, past_real, label='Past OT')
plt.plot(future_x, future_real, label='Ground Truth')
plt.plot(future_x, pred_real, label='Patch Transformer')
plt.axvline(0, linestyle='--')

plt.xlabel('Time Step')
plt.ylabel('OT (°C)')
plt.title('Direct Multi-Horizon Forecast')
plt.legend()
plt.grid(True)
plt.show()


## 12. 직접 해보기

- `PATCH_LEN`을 바꾸면 token 수가 어떻게 변하나요?
- `STRIDE`를 줄이면 patch overlap과 계산량은 어떻게 변하나요?
- Patch Transformer와 last-value baseline을 비교해 보세요.

## 핵심 정리

Patch를 사용하면 긴 시계열을 더 적은 token으로 표현할 수 있습니다.
이번 모델은 과거 96 step을 11개의 patch token으로 바꾸어 Transformer Encoder에 입력합니다.